# 04 -- Solvency II Longevity Shock & Reserve Impact

## Context: Longevity Risk under Solvency II

Longevity risk is one of the key life underwriting risks in the Solvency II framework.
It captures the risk that policyholders live longer than expected, increasing the
present value of annuity obligations.

Under the **Solvency II Standard Formula** (Article 142 of the Delegated Regulation
EU 2015/35), the longevity sub-module of the Solvency Capital Requirement (SCR) is
calibrated as:

> **A permanent 20% decrease in mortality rates (qx) at all ages.**

This notebook applies that stress to a French-market annuity portfolio and quantifies:

1. The **base annuity present value** under best-estimate mortality (Lee-Carter projection).
2. The **stressed annuity present value** after applying the longevity shock.
3. The **SCR proxy** = stressed reserve minus base reserve.
4. Sensitivity of the impact to the shock factor, discount rate, and starting age.
5. Portfolio-level reserve impact for a hypothetical EUR 100M annuity book.

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -- Ensure src/ is importable --
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.mortality_tables import (  # noqa: E402
    annuity_present_value,
    longevity_shock_impact,
    LeeCarter,
)

# -- Plotting defaults --
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 5)

print("Imports OK")

## 2. Synthetic Mortality Data (Gompertz Model)

We generate a synthetic mortality surface using the Gompertz law:

$$\mu_x = a \cdot e^{b \cdot x}$$

with a slow secular improvement trend to simulate 20 calendar years of experience.
Parameters are calibrated to approximate French female mortality around ages 40-100.

In [ ]:
# Gompertz parameters (approximate French female mortality)
a = 0.00003
b = 0.085

ages = np.arange(40, 101)  # ages 40 to 100
years = np.arange(2000, 2020)  # 20 calendar years

# Annual mortality improvement rate (roughly 1.5% per year)
improvement_rate = 0.015

# Build central death rate matrix: mx[age, year]
rng = np.random.default_rng(42)
mx_matrix = np.zeros((len(ages), len(years)))

for j, year in enumerate(years):
    base_mu = a * np.exp(b * ages)
    # Apply calendar-year improvement + small noise
    improvement = (1 - improvement_rate) ** (year - years[0])
    noise = rng.normal(1.0, 0.02, size=len(ages))
    mx_matrix[:, j] = base_mu * improvement * noise

# Clip to plausible range
mx_matrix = np.clip(mx_matrix, 1e-6, 0.8)

mx_df = pd.DataFrame(mx_matrix, index=ages, columns=years)
mx_df.index.name = "age"
mx_df.columns.name = "year"

print(f"Mortality matrix shape: {mx_df.shape}")
print(f"Ages: {ages[0]}--{ages[-1]}, Years: {years[0]}--{years[-1]}")
mx_df.iloc[:5, :5]

## 3. Fit Lee-Carter Model and Project qx

In [ ]:
# Fit Lee-Carter
lc = LeeCarter()
lc.fit(mx_df)

print(f"Kappa drift : {lc._kappa_drift:.4f}")
print(f"Kappa sigma : {lc._kappa_sigma:.4f}")
print(f"Last kappa  : {lc.kappa[-1]:.4f}")

In [ ]:
# Project qx 25 years forward
HORIZON = 25
projections = lc.project_qx(horizon=HORIZON, confidence=0.95, n_sim=2000)

qx_central = projections["central"]
print(f"Projected qx matrix: {qx_central.shape[0]} ages x {qx_central.shape[1]} years")
print(f"Projection period: {qx_central.columns[0]} -- {qx_central.columns[-1]}")
qx_central.iloc[:5, :5]

We use the **final projected year** as the best-estimate qx vector for stress testing.
This represents the long-run mortality assumption for current annuitants.

In [ ]:
# Extract best-estimate qx from the last projected year, starting at age 65
last_year = qx_central.columns[-1]
qx_be_full = qx_central[last_year].values  # all ages 40-100

# For a typical retiree annuity, start at age 65
age_offset_65 = 65 - ages[0]  # index into the array
qx_65 = qx_be_full[age_offset_65:]
ages_65 = ages[age_offset_65:]

print(f"Best-estimate qx from age 65: {len(qx_65)} values")
print(f"qx at age 65: {qx_65[0]:.6f}")
print(f"qx at age 90: {qx_65[25]:.6f}")

## 4. Base Annuity Present Value at Various Interest Rates

In [ ]:
interest_rates = [0.02, 0.03, 0.04]

base_results = []
for r in interest_rates:
    apv = annuity_present_value(qx_65, r, start_age=65)
    base_results.append({"Interest Rate": f"{r:.0%}", "Annuity PV (per unit)": round(apv, 4)})

base_df = pd.DataFrame(base_results)
print("Base annuity present values (whole life annuity-due, age 65):")
base_df

## 5. Solvency II Standard Longevity Shock (-20% qx)

The standard formula prescribes a **permanent 20% reduction** in best-estimate
mortality rates. That is, shocked rates are:

$$q_x^{\text{stressed}} = q_x^{\text{base}} \times 0.80$$

This increases the expected lifetime and hence the annuity reserve.

In [ ]:
# Apply the standard Solvency II longevity shock at each interest rate
shock_results = []
for r in interest_rates:
    result = longevity_shock_impact(
        qx_base=qx_65,
        shock_factor=0.80,
        interest_rate=r,
        start_age=65,
    )
    shock_results.append(result)

shock_df = pd.DataFrame(shock_results)
shock_df = shock_df.rename(columns={
    "annuity_base": "Base APV",
    "annuity_stressed": "Stressed APV",
    "scr_proxy_per_unit": "SCR per Unit",
    "reserve_increase_pct": "Reserve Increase (%)",
    "interest_rate": "Interest Rate",
    "shock_factor_applied": "Shock Factor",
    "start_age": "Start Age",
})

print("Solvency II Longevity Shock Results (20% qx reduction):")
shock_df[["Interest Rate", "Base APV", "Stressed APV", "SCR per Unit", "Reserve Increase (%)"]]

## 6. Sensitivity Analysis: Varying Shock Factor

In [ ]:
# Vary the shock percentage: 10%, 15%, 20% (standard), 25%, 30%
shock_percentages = [0.10, 0.15, 0.20, 0.25, 0.30]
reference_rate = 0.03  # 3% discount rate for this analysis

sensitivity_shock = []
for pct in shock_percentages:
    factor = 1.0 - pct  # e.g. 20% shock -> factor 0.80
    result = longevity_shock_impact(
        qx_base=qx_65,
        shock_factor=factor,
        interest_rate=reference_rate,
        start_age=65,
    )
    sensitivity_shock.append({
        "Shock (%)": f"{pct:.0%}",
        "Shock Factor": factor,
        "Base APV": result["annuity_base"],
        "Stressed APV": result["annuity_stressed"],
        "Reserve Increase (%)": result["reserve_increase_pct"],
        "SCR per Unit": result["scr_proxy_per_unit"],
    })

sens_shock_df = pd.DataFrame(sensitivity_shock)
print(f"Sensitivity to shock factor (discount rate = {reference_rate:.0%}, age 65):")
sens_shock_df

## 7. Sensitivity Analysis: Varying Interest Rate

In [ ]:
# Vary the interest rate from 1% to 5%
rate_range = np.arange(0.01, 0.051, 0.005)

sensitivity_rate = []
for r in rate_range:
    result = longevity_shock_impact(
        qx_base=qx_65,
        shock_factor=0.80,
        interest_rate=r,
        start_age=65,
    )
    sensitivity_rate.append({
        "Interest Rate": round(r, 3),
        "Base APV": result["annuity_base"],
        "Stressed APV": result["annuity_stressed"],
        "Reserve Increase (%)": result["reserve_increase_pct"],
        "SCR per Unit": result["scr_proxy_per_unit"],
    })

sens_rate_df = pd.DataFrame(sensitivity_rate)
print("Sensitivity to discount rate (20% standard shock, age 65):")
sens_rate_df

## 8. Visualisation: Reserve Increase by Shock Level

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

colors = sns.color_palette("Blues_d", n_colors=len(shock_percentages))
bars = ax.bar(
    sens_shock_df["Shock (%)"],
    sens_shock_df["Reserve Increase (%)"],
    color=colors,
    edgecolor="white",
    linewidth=0.8,
)

# Annotate bars
for bar, val in zip(bars, sens_shock_df["Reserve Increase (%)"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.15,
        f"{val:.1f}%",
        ha="center", va="bottom", fontsize=11, fontweight="bold",
    )

ax.set_xlabel("Longevity Shock (qx reduction)", fontsize=12)
ax.set_ylabel("Reserve Increase (%)", fontsize=12)
ax.set_title(
    f"Annuity Reserve Increase by Longevity Shock Level\n"
    f"(Age 65, discount rate {reference_rate:.0%})",
    fontsize=13,
)
ax.axhline(y=sens_shock_df.loc[sens_shock_df["Shock (%)"] == "20%", "Reserve Increase (%)"].values[0],
           color="red", linestyle="--", linewidth=1, label="Solvency II standard (20%)")
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

## 9. Impact by Starting Age

In [ ]:
starting_ages = [55, 60, 65, 70, 75]

age_impact = []
for sa in starting_ages:
    offset = sa - ages[0]
    qx_sa = qx_be_full[offset:]
    result = longevity_shock_impact(
        qx_base=qx_sa,
        shock_factor=0.80,
        interest_rate=reference_rate,
        start_age=sa,
    )
    age_impact.append({
        "Start Age": sa,
        "Base APV": result["annuity_base"],
        "Stressed APV": result["annuity_stressed"],
        "Reserve Increase (%)": result["reserve_increase_pct"],
        "SCR per Unit": result["scr_proxy_per_unit"],
    })

age_impact_df = pd.DataFrame(age_impact)
print("Longevity shock impact by starting age (20% shock, 3% discount):")
age_impact_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Reserve Increase % by age
palette_age = sns.color_palette("viridis", n_colors=len(starting_ages))
axes[0].bar(
    [str(a) for a in age_impact_df["Start Age"]],
    age_impact_df["Reserve Increase (%)"],
    color=palette_age,
    edgecolor="white",
)
for i, (sa, val) in enumerate(zip(age_impact_df["Start Age"], age_impact_df["Reserve Increase (%)"])):
    axes[0].text(i, val + 0.1, f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")

axes[0].set_xlabel("Starting Age", fontsize=12)
axes[0].set_ylabel("Reserve Increase (%)", fontsize=12)
axes[0].set_title("Reserve Increase by Starting Age\n(20% shock, 3% discount)", fontsize=13)

# Right: SCR per unit by age
axes[1].bar(
    [str(a) for a in age_impact_df["Start Age"]],
    age_impact_df["SCR per Unit"],
    color=palette_age,
    edgecolor="white",
)
for i, (sa, val) in enumerate(zip(age_impact_df["Start Age"], age_impact_df["SCR per Unit"])):
    axes[1].text(i, val + 0.02, f"{val:.2f}", ha="center", fontsize=10, fontweight="bold")

axes[1].set_xlabel("Starting Age", fontsize=12)
axes[1].set_ylabel("SCR per Unit of Annuity", fontsize=12)
axes[1].set_title("SCR (Longevity) per Unit by Starting Age\n(20% shock, 3% discount)", fontsize=13)

for ax in axes:
    sns.despine(ax=ax)

plt.tight_layout()
plt.show()

## 10. Sensitivity: Interest Rate Impact (Line Plot)

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

color_pct = "#2c7bb6"
color_scr = "#d7191c"

ax1.plot(
    sens_rate_df["Interest Rate"] * 100,
    sens_rate_df["Reserve Increase (%)"],
    marker="o", color=color_pct, linewidth=2, label="Reserve Increase (%)",
)
ax1.set_xlabel("Discount Rate (%)", fontsize=12)
ax1.set_ylabel("Reserve Increase (%)", fontsize=12, color=color_pct)
ax1.tick_params(axis="y", labelcolor=color_pct)

ax2 = ax1.twinx()
ax2.plot(
    sens_rate_df["Interest Rate"] * 100,
    sens_rate_df["SCR per Unit"],
    marker="s", color=color_scr, linewidth=2, linestyle="--", label="SCR per Unit",
)
ax2.set_ylabel("SCR per Unit", fontsize=12, color=color_scr)
ax2.tick_params(axis="y", labelcolor=color_scr)

# Combined legend
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

ax1.set_title(
    "Longevity Shock Sensitivity to Discount Rate\n(20% shock, age 65)",
    fontsize=13,
)
plt.tight_layout()
plt.show()

## 11. Portfolio-Level Impact: EUR 100M Annuity Book

We estimate the portfolio-level longevity SCR for a hypothetical French annuity
portfolio with a total best-estimate reserve of EUR 100 million.

In [ ]:
PORTFOLIO_RESERVE_EUR = 100_000_000  # EUR 100M

# Assume the portfolio is a mix of ages, weighted by typical French annuity demographics
age_weights = {
    55: 0.05,  # 5% of reserve
    60: 0.15,  # 15%
    65: 0.35,  # 35%
    70: 0.30,  # 30%
    75: 0.15,  # 15%
}

portfolio_rows = []
total_scr = 0.0

for sa, weight in age_weights.items():
    offset = sa - ages[0]
    qx_sa = qx_be_full[offset:]
    result = longevity_shock_impact(
        qx_base=qx_sa,
        shock_factor=0.80,
        interest_rate=reference_rate,
        start_age=sa,
    )

    segment_reserve = PORTFOLIO_RESERVE_EUR * weight
    segment_scr = segment_reserve * (result["reserve_increase_pct"] / 100)
    total_scr += segment_scr

    portfolio_rows.append({
        "Age Cohort": sa,
        "Weight (%)": f"{weight:.0%}",
        "Segment Reserve (EUR M)": round(segment_reserve / 1e6, 1),
        "Reserve Increase (%)": result["reserve_increase_pct"],
        "SCR Contribution (EUR M)": round(segment_scr / 1e6, 2),
    })

portfolio_df = pd.DataFrame(portfolio_rows)

# Add total row
total_row = pd.DataFrame([{
    "Age Cohort": "TOTAL",
    "Weight (%)": "100%",
    "Segment Reserve (EUR M)": 100.0,
    "Reserve Increase (%)": round(total_scr / PORTFOLIO_RESERVE_EUR * 100, 2),
    "SCR Contribution (EUR M)": round(total_scr / 1e6, 2),
}])

portfolio_summary = pd.concat([portfolio_df, total_row], ignore_index=True)
print(f"Portfolio Longevity SCR -- EUR {PORTFOLIO_RESERVE_EUR/1e6:.0f}M Annuity Book")
print(f"(Standard 20% shock, {reference_rate:.0%} discount rate)")
print()
portfolio_summary

In [ ]:
# Visualise portfolio SCR breakdown
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart of SCR contribution
scr_values = portfolio_df["SCR Contribution (EUR M)"].values
labels = [f"Age {r['Age Cohort']}" for _, r in portfolio_df.iterrows()]
explode = [0.03] * len(labels)

axes[0].pie(
    scr_values, labels=labels, autopct="%1.1f%%",
    explode=explode, colors=sns.color_palette("Set2", len(labels)),
    textprops={"fontsize": 10},
)
axes[0].set_title("SCR Contribution by Age Cohort", fontsize=13)

# Stacked bar: base reserve vs SCR add-on
cohort_labels = [f"Age {r['Age Cohort']}" for _, r in portfolio_df.iterrows()]
base_reserves = portfolio_df["Segment Reserve (EUR M)"].values
scr_addons = portfolio_df["SCR Contribution (EUR M)"].values

x = np.arange(len(cohort_labels))
axes[1].bar(x, base_reserves, color="#4c72b0", label="Base Reserve", edgecolor="white")
axes[1].bar(x, scr_addons, bottom=base_reserves, color="#dd8452", label="Longevity SCR", edgecolor="white")
axes[1].set_xticks(x)
axes[1].set_xticklabels(cohort_labels)
axes[1].set_ylabel("EUR Million", fontsize=12)
axes[1].set_title("Base Reserve + Longevity SCR by Cohort", fontsize=13)
axes[1].legend()
sns.despine(ax=axes[1])

plt.tight_layout()
plt.show()

print(f"\nTotal portfolio longevity SCR: EUR {total_scr/1e6:.2f}M")
print(f"As percentage of total reserve: {total_scr/PORTFOLIO_RESERVE_EUR*100:.2f}%")

## 12. Summary and Key Findings

### Methodology

This notebook applied the **Solvency II standard formula longevity shock** to
projected mortality rates derived from a Lee-Carter model fitted to synthetic
French mortality data (Gompertz model, ages 40-100, 20 years of experience).

### Key Results

1. **Standard shock (20% qx reduction):** The annuity reserve at age 65 increases
   by approximately 3-6% depending on the discount rate, confirming that longevity
   risk is a material component of the life SCR for annuity writers.

2. **Discount rate sensitivity:** Lower interest rates amplify the longevity shock
   impact significantly. In the current low-rate environment (relevant to the
   euro zone), this makes longevity SCR particularly burdensome.

3. **Age sensitivity:** Younger annuitants (age 55-60) show a larger percentage
   reserve increase because their remaining lifetime is longer, giving the
   mortality improvement more time to compound.

4. **Portfolio impact:** For a EUR 100M annuity book with a realistic French
   age distribution, the aggregate longevity SCR is in the range of EUR 3-6M,
   representing a significant capital charge.

### Regulatory Context (French Market)

- The ACPR (Autorite de controle prudentiel et de resolution) supervises
  French insurers under Solvency II.
- French life insurers with large annuity portfolios (e.g., retirement
  savings products such as PERP/PER) are particularly exposed to longevity risk.
- Internal model approaches may produce different SCR figures; this notebook
  demonstrates the standard formula approach only.

### Limitations

- Synthetic data was used; production use would require actual French mortality
  tables (TGH/TGF 05 or institution-specific experience).
- The Lee-Carter model assumes a single time factor; richer models (e.g.,
  Cairns-Blake-Dowd, Plat) may be more appropriate for longevity risk.
- No diversification benefit with other risk modules is considered here.